In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# =========================
# 1. Load dataset
# =========================
df = pd.read_csv("Melbourne_Housing_Model_Input.csv")

df.columns = df.columns.str.strip()

df = df.rename(columns={
    "Property_Type": "Type"
})

print("Original rows:", len(df))
print("Columns:", df.columns.tolist())


# =========================
# 2. Basic cleaning
# =========================
df["Sales_ID"] = pd.to_numeric(df["Sales_ID"], errors="coerce")
df = df.dropna(subset=["Sales_ID"])
df["Sales_ID"] = df["Sales_ID"].astype(int)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Clean currency column
df["Price"] = (
    df["Price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# Clean BuildingArea
df["BuildingArea"] = (
    df["BuildingArea"]
    .astype(str)
    .replace(["missing", "inf", "Infinity", "-inf", "nan", "None"], np.nan)
    .str.replace(",", "", regex=False)
    .str.strip()
)

numeric_cols = [
    "Rooms",
    "Distance",
    "Bedroom",
    "Bathroom",
    "Car",
    "Landsize",
    "BuildingArea",
    "YearBuilt",
    "Propertycount",
    "Price"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["Price"])

print("Rows after cleaning:", len(df))


# =========================
# 3. Feature selection
# =========================
numeric_features = [
    "Rooms",
    "Bedroom",
    "Bathroom",
    "Car",
    "Landsize",
    "BuildingArea",
    "YearBuilt",
    "Distance",
    "Propertycount"
]

categorical_features = [
    "Type",
    "Regionname"
]

target = "Price"


# =========================
# 4. Define model scope
# =========================
# These filters define rows that are reasonable for model training and prediction.
# Extreme rows remain in the output but are marked as Out of Model Scope.

df["InModelScope"] = (
    (df["Price"] >= 100000) &
    (df["Price"] <= 5000000) &
    (df["Rooms"].isna() | ((df["Rooms"] >= 1) & (df["Rooms"] <= 8))) &
    (df["Bedroom"].isna() | ((df["Bedroom"] >= 1) & (df["Bedroom"] <= 8))) &
    (df["Bathroom"].isna() | ((df["Bathroom"] >= 1) & (df["Bathroom"] <= 6))) &
    (df["Car"].isna() | ((df["Car"] >= 0) & (df["Car"] <= 6))) &
    (df["Landsize"].isna() | ((df["Landsize"] > 0) & (df["Landsize"] <= 5000))) &
    (df["BuildingArea"].isna() | ((df["BuildingArea"] > 0) & (df["BuildingArea"] <= 1000)))
)

df["ModelScope"] = np.where(
    df["InModelScope"],
    "In Model Scope",
    "Out of Model Scope"
)

model_df = df[df["InModelScope"]].copy()

print("Rows in model scope:", len(model_df))
print("Rows out of model scope:", len(df) - len(model_df))

if len(model_df) == 0:
    raise ValueError("No rows available for model training. Check cleaning and scope filters.")


# =========================
# 5. Prepare model data
# =========================
X = model_df[numeric_features + categorical_features]
y = model_df[target]


# =========================
# 6. Preprocessing
# =========================
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


# =========================
# 7. Random Forest model
# =========================
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=18,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


# =========================
# 8. Train-test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)


# =========================
# 9. Evaluation
# =========================
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nRandom Forest Model Performance")
print(f"MAE:  ${mae:,.0f}")
print(f"RMSE: ${rmse:,.0f}")
print(f"R²:   {r2:.3f}")


# =========================
# 10. Predict only in-scope rows
# =========================
df["PredictedPrice"] = np.nan

in_scope_mask = df["InModelScope"]

X_in_scope = df.loc[in_scope_mask, numeric_features + categorical_features]
df.loc[in_scope_mask, "PredictedPrice"] = model.predict(X_in_scope)

# Optional: keep predictions within a realistic range based on training data
min_train_price = model_df["Price"].quantile(0.01)
max_train_price = model_df["Price"].quantile(0.99)

df.loc[in_scope_mask, "PredictedPrice"] = df.loc[in_scope_mask, "PredictedPrice"].clip(
    lower=min_train_price,
    upper=max_train_price
)


# =========================
# 11. Value gap and recommendation
# =========================
df["ValueGap"] = df["PredictedPrice"] - df["Price"]
df["ValueGapPercentage"] = df["ValueGap"] / df["Price"]
df["AbsoluteError"] = abs(df["Price"] - df["PredictedPrice"])


def recommendation_category(row):
    if row["ModelScope"] == "Out of Model Scope":
        return "Out of Model Scope"

    if pd.isna(row["ValueGapPercentage"]):
        return "No Prediction"

    if row["ValueGapPercentage"] >= 0.15:
        return "Potentially Undervalued"
    elif row["ValueGapPercentage"] <= -0.15:
        return "Potentially Overvalued"
    else:
        return "Fairly Priced"


df["RecommendationCategory"] = df.apply(recommendation_category, axis=1)


# =========================
# 12. Export prediction output
# =========================
output_cols = [
    "Sales_ID",
    "Date",
    "Suburb",
    "Regionname",
    "Type",
    "Rooms",
    "Bedroom",
    "Bathroom",
    "Car",
    "Landsize",
    "BuildingArea",
    "YearBuilt",
    "Distance",
    "Propertycount",
    "Price",
    "PredictedPrice",
    "ValueGap",
    "ValueGapPercentage",
    "AbsoluteError",
    "RecommendationCategory",
    "ModelScope"
]

prediction_output = df[output_cols].copy()

prediction_output.to_csv(
    "Melbourne_Housing_Predictions_RF.csv",
    index=False
)


# =========================
# 13. Export model metrics
# =========================
metrics_df = pd.DataFrame({
    "Metric": [
        "Model",
        "MAE",
        "RMSE",
        "R2",
        "InputRows",
        "PredictionRows",
        "TrainingRows",
        "OutOfModelScopeRows"
    ],
    "Value": [
        "Random Forest Regressor",
        mae,
        rmse,
        r2,
        len(df),
        len(prediction_output),
        len(model_df),
        len(df) - len(model_df)
    ]
})

metrics_df.to_csv(
    "Melbourne_Housing_Model_Metrics_RF.csv",
    index=False
)


# =========================
# 14. ID check
# =========================
input_ids = set(df["Sales_ID"])
prediction_ids = set(prediction_output["Sales_ID"])

print("\nFiles exported:")
print("Melbourne_Housing_Predictions_RF.csv")
print("Melbourne_Housing_Model_Metrics_RF.csv")

print("\nID check:")
print("Input rows:", len(df))
print("Prediction rows:", len(prediction_output))
print("Missing Sales_ID:", len(input_ids - prediction_ids))
print("Extra Sales_ID:", len(prediction_ids - input_ids))

print("\nRecommendation category counts:")
print(df["RecommendationCategory"].value_counts(dropna=False))

Original rows: 27103
Columns: ['Sales_ID', 'Date', 'Suburb', 'Regionname', 'Type', 'Rooms', 'Bedroom', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'Price', 'Distance', 'Propertycount']
Rows after cleaning: 27103
Rows in model scope: 26859
Rows out of model scope: 244

Random Forest Model Performance
MAE:  $171,640
RMSE: $276,384
R²:   0.784

Files exported:
Melbourne_Housing_Predictions_RF.csv
Melbourne_Housing_Model_Metrics_RF.csv

ID check:
Input rows: 27103
Prediction rows: 27103
Missing Sales_ID: 0
Extra Sales_ID: 0

Recommendation category counts:
RecommendationCategory
Fairly Priced              18164
Potentially Undervalued     5708
Potentially Overvalued      2987
Out of Model Scope           244
Name: count, dtype: int64
